In [1]:
!ls

data_explore.ipynb       recipe_summaries.parquet


In [2]:
import pandas as pd
data = pd.read_parquet("recipe_summaries.parquet")

In [3]:
data.columns

Index(['flow_id', 'version_no', 'author_id', 'connectors', 'step_count',
       'recipe_summary_with_comment', 'recipe_summary_without_comment'],
      dtype='str')

In [7]:
print(data[data['flow_id']==53594265]["recipe_summary_without_comment"].values[0])

Connectors: airtable, email, salesforce
Steps:
- trigger: airtable / new_or_updated_record
  - try [error handling]
    - action: salesforce / search_sobjects
      fields: sobject_name, limit, Id, field_list
    - if [and: greater_than]
      - try [error handling]
        - action: salesforce / update_sobject
          fields: sobject_name, id, At_Risk_Reason__c, At_Risk_Status__c, Last_QBR_Done_On__c, Last_Onsite_Visit__c, Customer_Live__c, Embedded_Type__c, (+1 more)
        - action: airtable / update_record
          fields: base_id, table_id, id, fldZQnXwj8GAfTXBu, fld6KOzfxgdmMBh2j
        - catch [error handler]
          - action: airtable / update_record
            fields: base_id, table_id, id, fldZQnXwj8GAfTXBu
    - catch [error handler]
      - action: email / send_mail
        fields: email_type, to, subject, body
      - stop


In [13]:
import pandas as pd
from pathlib import Path

# Absolute path so it works regardless of kernel working directory
EVAL_DIR = Path("/Users/alice/project/semantic_search/search_engine/pipeline/03_evaluate_embeddings")

print("CWD:", Path.cwd())
hybrid = pd.read_csv(EVAL_DIR / "hybrid_english+Qwen_Qwen3_Embedding_8B+instruct_category3_k5.csv")
print(hybrid.shape)
hybrid.head()


CWD: /Users/alice/project/semantic_search/search_engine/pipeline/01_process_data/cleaned
(49, 14)


,category,query_id,query,n_strong,n_weak,precision@5,recall@5,rr,strong_hits_top5,weak_hits_top5,n_retrieved,retrieved,strong_list,weak_list
0,Category 3,8621_c3q20,If the workato_db_table get_records action cha...,6,2,0.0,0.0,0.0,0,0,5,"384059_52352657_v71, 804586_47150941_v29, 6024...","136763_23098236_v2, 136763_61448486_v1, 136763...","3000083_51901364_v24, 973497_45184852_v4"
1,Category 3,8621_c3q5,Which recipes reference the workato_internal_a...,1,0,0.2,1.0,1.0,1,0,5,"8621_45178477_v1, 2436037_45178430_v28, 187334...",8621_45178477_v1,NaN
2,Category 3,15806_c3q1,Which recipes reference the workato_finance_wo...,1,0,0.2,1.0,0.5,1,0,5,"136763_23099454_v8, 15806_57106439_v10, 384059...",15806_57106439_v10,NaN
3,Category 3,50737_c3q12,Which recipes depend on the corporate_api_conn...,1,5,0.0,0.0,0.0,0,0,5,"1873346_55341286_v18, 50737_45179043_v13, 1873...",50737_46237411_v22,"15806_57106439_v10, 1672048_15022103_v41, 3840..."
4,Category 3,69943_c3q10,Which recipes depend on the Jira __adhoc_http_...,1,0,0.2,1.0,0.5,1,0,5,"8621_51493003_v37, 69943_46962800_v13, 2366099...",69943_46962800_v13,NaN


In [17]:
hybrid[hybrid["query_id"]=="1873346_c3q14"]

,category,query_id,query,n_strong,n_weak,precision@5,recall@5,rr,strong_hits_top5,weak_hits_top5,n_retrieved,retrieved,strong_list,weak_list
37,Category 3,1873346_c3q14,If the Salesforce update_sobject action updati...,1,2,0.2,1.0,1.0,1,0,5,"1873346_53594265_v8, 8621_45073029_v1, 1873346...",1873346_53594265_v8,"800219_23098652_v2, 8621_45178477_v1"


In [16]:
hybrid[hybrid["query_id"]=="1873346_c3q14"]["retrieved"].values

<ArrowStringArray>
['1873346_53594265_v8, 8621_45073029_v1, 1873346_63519497_v1, 136763_23099454_v8, 800219_23099317_v1']
Length: 1, dtype: str

In [21]:
print(data[data['flow_id']==23099317]["recipe_summary_without_comment"].values[0])

Connectors: lookup_table, salesforce, workato_recipe_function
Steps:
- trigger: workato_recipe_function / execute
  - try [error handling]
    - if [or: present, present, present, present, present]
      - action: lookup_table / search_entries
        fields: lookup_table_id, parameters
      - if [and: greater_than]
        - foreach [loop]
          - action: lookup_table / delete_entry
            fields: ignore_not_found, id, lookup_table_id
      - if [or: equals_to]
        - action: lookup_table / add_entry
          fields: lookup_table_id, parameters
      - action: lookup_table / update_entry
        fields: ignore_not_found, lookup_table_id, id, parameters
    - action: salesforce / search_sobjects
      fields: limit, sobject_name, field_list, Email__c
    - if [and: blank]
      - action: salesforce / create_custom_object
        fields: sobject_name, Email__c, Name
    - try [error handling]
      - action: salesforce / update_sobject
        fields: sobject_name, id, Nam